In [20]:
import pandas as pd

chomage=pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\tx_chom.csv", sep=";")

chomage = chomage.head(15)
chomage['Taux de chômage par région'] = chomage['Taux de chômage par région'].str.replace('Taux de chômage localisé par région - ', '')
#renommer la colonne Taux de chômage par région par region_nom
chomage = chomage.rename(columns={'Taux de chômage par région': 'region_nom'})

In [21]:
import re

# -----------------------------------------------------------------------

# 2. Transformation (Pivot -> Unpivot)
# id_vars = la colonne qui identifie la ligne (la région)
# var_name = comment appeler la colonne qui contiendra les anciens noms de colonnes
# value_name = comment appeler la colonne qui contiendra les valeurs
chomage1 = chomage.melt(id_vars=['region_nom'], 
                  var_name='colonne_temporaire', 
                  value_name='Taux de chômage par région')

# À ce stade, 'colonne_temporaire' contient "Moyenne_2015", "Moyenne_2016"...
# On doit extraire juste l'année.

# 3. Extraction de l'année (TIME_VALUE)
# La regex r'(\d{4})' cherche une suite de 4 chiffres dans le texte
chomage1['TIME_VALUE'] = chomage1['colonne_temporaire'].astype(str).str.extract(r'(\d{4})').astype(int)

# 4. Nettoyage final
# On supprime la colonne temporaire qui ne sert plus
chomage_format_long = chomage1.drop(columns=['colonne_temporaire'])

# On renomme 'region_nom' en 'Region' pour standardiser (optionnel mais conseillé)
chomage_format_long.rename(columns={'region_nom': 'Region'}, inplace=True)

# On trie pour avoir une lecture logique (Par Région puis par Année)
chomage_format_long = chomage_format_long.sort_values(by=['Region', 'TIME_VALUE']).reset_index(drop=True)

# Sauvegarde
chomage_format_long.to_csv(r"C:\Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives/chomage_format_long.csv", index=False)

In [25]:
creation_per_1000 = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\creation_per_1000.csv") 
#colonne TIME_PERIOD en TIME_VALUE
creation_per_1000 = creation_per_1000.rename(columns={'TIME_PERIOD': 'TIME_VALUE'})
chomage_format_long = chomage_format_long.rename(columns={'Region': 'region_nom'})

In [26]:
df_merged = pd.merge(chomage_format_long, creation_per_1000, on=['region_nom', 'TIME_VALUE'], how='inner')

